Computational Complexity — Baseline and Attention Mechanisms
This notebook evaluates the computational complexity of the five CNN configurations used in the study:
- Baseline CNN
- CNN + CBAM
- CNN + CoordAttention
- CNN + Triplet Attention
- CNN + ECA-Net
It does not require the BUSI/BUS-BRA datasets, trained weights, or the original experimental notebooks.
It calculates total parameters, parameters added relative to the baseline, percentage increase, FLOPs/GFLOPs, inference time, and throughput (FPS).
Google Colab
1. Open this notebook in Google Colab.
2. For latency/FPS measurements, select Runtime → Change runtime type → T4 GPU.
3. Select Runtime → Run all.
4. Send the final table/output back to ChatGPT for verification before using the values in the manuscript.

In [1]:
# ============================================================
# COMPLEXIDADE COMPUTACIONAL - TODOS OS MODELOS
# Baseline, CBAM, CoordAttention, Triplet Attention e ECA-Net
#
# Extraído das arquiteturas dos notebooks enviados.
# Não requer treinamento, pesos salvos nem dataset.
# Entrada: 224 x 224 x 1
# ============================================================

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

IMG_SZ = 224
NUM_WARMUP = 20
NUM_RUNS = 100

print("TensorFlow:", tf.__version__)
print("Dispositivos:", tf.config.list_physical_devices())
print()


# ============================================================
# BLOCOS DE ATENÇÃO
# ============================================================

# ---------------- CBAM ----------------
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    shared_layer_one = layers.Dense(
        channel // ratio,
        activation="relu",
        kernel_initializer="he_normal",
        use_bias=True,
        bias_initializer="zeros",
    )
    shared_layer_two = layers.Dense(
        channel,
        kernel_initializer="he_normal",
        use_bias=True,
        bias_initializer="zeros",
    )

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_layer_one(avg_pool)
    avg_pool = shared_layer_two(avg_pool)

    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)
    max_pool = shared_layer_one(max_pool)
    max_pool = shared_layer_two(max_pool)

    cbam_feature = layers.Add()([avg_pool, max_pool])
    cbam_feature = layers.Activation("sigmoid")(cbam_feature)

    return layers.Multiply()([input_feature, cbam_feature])


def spatial_attention(input_feature):
    avg_pool = layers.Lambda(
        lambda x: tf.reduce_mean(x, axis=-1, keepdims=True)
    )(input_feature)

    max_pool = layers.Lambda(
        lambda x: tf.reduce_max(x, axis=-1, keepdims=True)
    )(input_feature)

    concat = layers.Concatenate(axis=-1)([avg_pool, max_pool])

    cbam_feature = layers.Conv2D(
        filters=1,
        kernel_size=7,
        strides=1,
        padding="same",
        activation="sigmoid",
        kernel_initializer="he_normal",
        use_bias=False,
    )(concat)

    return layers.Multiply()([input_feature, cbam_feature])


def cbam_block(input_feature, ratio=4):
    feature = channel_attention(input_feature, ratio)
    feature = spatial_attention(feature)
    return feature


# ---------------- CoordAttention ----------------
def coord_attention(x, reduction=32):
    input_channels = x.shape[-1]

    h_pool = layers.Lambda(
        lambda z: tf.reduce_mean(z, axis=2, keepdims=True)
    )(x)

    w_pool = layers.Lambda(
        lambda z: tf.reduce_mean(z, axis=1, keepdims=True)
    )(x)

    w_pool = layers.Lambda(
        lambda z: tf.transpose(z, perm=[0, 2, 1, 3])
    )(w_pool)

    y = layers.Concatenate(axis=1)([h_pool, w_pool])

    reduced_channels = max(8, input_channels // reduction)

    y = layers.Conv2D(
        reduced_channels,
        kernel_size=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
    )(y)

    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)

    h, w = layers.Lambda(
        lambda z: tf.split(z, num_or_size_splits=2, axis=1)
    )(y)

    w = layers.Lambda(
        lambda z: tf.transpose(z, perm=[0, 2, 1, 3])
    )(w)

    h_att = layers.Conv2D(
        input_channels,
        kernel_size=1,
        padding="same",
        activation="sigmoid",
        use_bias=False,
        kernel_initializer="he_normal",
    )(h)

    w_att = layers.Conv2D(
        input_channels,
        kernel_size=1,
        padding="same",
        activation="sigmoid",
        use_bias=False,
        kernel_initializer="he_normal",
    )(w)

    out = layers.Multiply()([x, h_att])
    out = layers.Multiply()([out, w_att])

    return out


# ---------------- Triplet Attention ----------------
def basic_conv(x):
    return layers.Conv2D(
        filters=1,
        kernel_size=7,
        padding="same",
        activation="sigmoid",
        kernel_initializer="he_normal",
        use_bias=False,
    )(x)


def triplet_attention(x):
    # Branch 1: H x W
    x1 = layers.Lambda(
        lambda z: tf.reduce_mean(z, axis=-1, keepdims=True)
    )(x)
    x1 = basic_conv(x1)
    out1 = layers.Multiply()([x, x1])

    # Branch 2
    x_perm1 = layers.Permute((2, 1, 3))(x)
    x2 = layers.Lambda(
        lambda z: tf.reduce_mean(z, axis=-1, keepdims=True)
    )(x_perm1)
    x2 = basic_conv(x2)
    x2 = layers.Permute((2, 1, 3))(x2)
    out2 = layers.Multiply()([x, x2])

    # Branch 3
    x_perm2 = layers.Permute((3, 2, 1))(x)
    x3 = layers.Lambda(
        lambda z: tf.reduce_mean(z, axis=-1, keepdims=True)
    )(x_perm2)
    x3 = basic_conv(x3)
    x3 = layers.Permute((3, 2, 1))(x3)
    out3 = layers.Multiply()([x, x3])

    return layers.Average()([out1, out2, out3])


# ---------------- ECA-Net ----------------
def eca_block(input_feature, k_size=3):
    channel = input_feature.shape[-1]

    x = layers.GlobalAveragePooling2D()(input_feature)
    x = layers.Reshape((channel, 1))(x)

    x = layers.Conv1D(
        filters=1,
        kernel_size=k_size,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
    )(x)

    x = layers.Activation("sigmoid")(x)
    x = layers.Reshape((1, 1, channel))(x)

    return layers.Multiply()([input_feature, x])


# ============================================================
# CONSTRUTOR ÚNICO DOS MODELOS
# ============================================================

def build_model(attention=None):
    inputs = layers.Input(shape=(IMG_SZ, IMG_SZ, 1))

    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    if attention is not None:
        x = attention(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    if attention is not None:
        x = attention(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(
        128, (3, 3), dilation_rate=2, padding="same", activation="relu"
    )(x)
    if attention is not None:
        x = attention(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(
        256, (3, 3), dilation_rate=2, padding="same", activation="relu"
    )(x)
    if attention is not None:
        x = attention(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(2, activation="softmax")(x)

    return models.Model(inputs=inputs, outputs=outputs)


# ============================================================
# FLOPs
# ============================================================

def get_flops(model):
    input_shape = model.input_shape[1:]

    concrete_function = tf.function(
        lambda inputs: model(inputs, training=False)
    ).get_concrete_function(
        tf.TensorSpec([1] + list(input_shape), dtype=tf.float32)
    )

    frozen_func = convert_variables_to_constants_v2(concrete_function)
    graph = frozen_func.graph

    with graph.as_default():
        options = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        # Evita imprimir a listagem enorme do profiler.
        options["output"] = "none"

        profile = tf.compat.v1.profiler.profile(
            graph=graph,
            options=options,
        )

    return 0 if profile is None else profile.total_float_ops


# ============================================================
# TEMPO DE INFERÊNCIA
# Mesmo procedimento para TODOS os modelos.
# Usa a mesma entrada sintética 224x224x1, batch_size=1.
# ============================================================

def benchmark_inference(model, img_input):
    # Warm-up
    for _ in range(NUM_WARMUP):
        _ = model(img_input, training=False)

    # Sincronização quando disponível.
    # A conversão para NumPy força a conclusão da execução antes de parar
    # o cronômetro e torna a medição mais segura em GPU.
    inference_times = []

    for _ in range(NUM_RUNS):
        start = time.perf_counter()
        y = model(img_input, training=False)
        _ = y.numpy()
        end = time.perf_counter()

        inference_times.append(end - start)

    inference_times = np.asarray(inference_times, dtype=np.float64)

    mean_time = np.mean(inference_times)
    std_time = np.std(inference_times, ddof=1)
    median_time = np.median(inference_times)

    return {
        "mean_ms": mean_time * 1000.0,
        "std_ms": std_time * 1000.0,
        "median_ms": median_time * 1000.0,
        "fps": 1.0 / mean_time,
    }


# ============================================================
# CRIAÇÃO DOS CINCO MODELOS
# ============================================================

tf.keras.backend.clear_session()

model_builders = [
    ("Baseline CNN", None),
    ("CBAM", cbam_block),
    ("CoordAttention", coord_attention),
    ("Triplet Attention", triplet_attention),
    ("ECA-Net", eca_block),
]

# Uma única imagem é suficiente para medir a latência.
# O conteúdo da imagem não altera a complexidade da arquitetura.
tf.random.set_seed(123)
img_input = tf.random.uniform(
    (1, IMG_SZ, IMG_SZ, 1),
    minval=0.0,
    maxval=1.0,
    dtype=tf.float32,
)

results = []
baseline_params = None

for name, attention in model_builders:
    print("\n" + "=" * 72)
    print(name)
    print("=" * 72)

    model = build_model(attention)

    total_params = int(model.count_params())

    trainable_params = int(
        np.sum([np.prod(v.shape) for v in model.trainable_weights])
    )
    non_trainable_params = int(
        np.sum([np.prod(v.shape) for v in model.non_trainable_weights])
    )

    if baseline_params is None:
        baseline_params = total_params

    added_params = total_params - baseline_params
    increase_pct = (
        (added_params / baseline_params) * 100.0
        if baseline_params > 0 else 0.0
    )

    try:
        flops = int(get_flops(model))
        gflops = flops / 1e9
    except Exception as e:
        print("Erro ao calcular FLOPs:", e)
        flops = 0
        gflops = np.nan

    try:
        timing = benchmark_inference(model, img_input)
    except Exception as e:
        print("Erro ao medir inferência:", e)
        timing = {
            "mean_ms": np.nan,
            "std_ms": np.nan,
            "median_ms": np.nan,
            "fps": np.nan,
        }

    print(f"Parâmetros totais:         {total_params:,}")
    print(f"Parâmetros treináveis:     {trainable_params:,}")
    print(f"Parâmetros não treináveis: {non_trainable_params:,}")
    print(f"Parâmetros adicionados:    {added_params:,}")
    print(f"Aumento vs. baseline:      {increase_pct:.4f}%")
    print(f"FLOPs:                     {flops:,}")
    print(f"GFLOPs:                    {gflops:.4f}")
    print(
        f"Inferência:                "
        f"{timing['mean_ms']:.3f} ± {timing['std_ms']:.3f} ms/imagem"
    )
    print(f"Mediana:                   {timing['median_ms']:.3f} ms/imagem")
    print(f"Throughput:                {timing['fps']:.2f} FPS")

    results.append({
        "Model": name,
        "Total Parameters": total_params,
        "Added Parameters": added_params,
        "Increase (%)": increase_pct,
        "FLOPs": flops,
        "GFLOPs": gflops,
        "Inference Mean (ms)": timing["mean_ms"],
        "Inference SD (ms)": timing["std_ms"],
        "Median (ms)": timing["median_ms"],
        "FPS": timing["fps"],
    })

    # Libera o modelo antes de criar o próximo.
    del model


# ============================================================
# TABELA FINAL
# ============================================================

print("\n\n" + "=" * 120)
print("TABELA FINAL")
print("=" * 120)

header = (
    f"{'Model':<20}"
    f"{'Parameters':>14}"
    f"{'Added':>12}"
    f"{'Increase %':>13}"
    f"{'GFLOPs':>12}"
    f"{'Mean ms':>12}"
    f"{'SD ms':>12}"
    f"{'FPS':>10}"
)
print(header)
print("-" * len(header))

for r in results:
    print(
        f"{r['Model']:<20}"
        f"{r['Total Parameters']:>14,}"
        f"{r['Added Parameters']:>12,}"
        f"{r['Increase (%)']:>13.4f}"
        f"{r['GFLOPs']:>12.4f}"
        f"{r['Inference Mean (ms)']:>12.3f}"
        f"{r['Inference SD (ms)']:>12.3f}"
        f"{r['FPS']:>10.2f}"
    )

print("=" * 120)


# ============================================================
# CSV PARA COPIAR DIRETAMENTE PARA O ARTIGO/PLANILHA
# ============================================================

try:
    import csv

    output_file = "computational_complexity_all_models.csv"

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    print(f"\nArquivo CSV salvo em: {output_file}")

except Exception as e:
    print("\nNão foi possível salvar o CSV:", e)


TensorFlow: 2.20.0
Dispositivos: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Baseline CNN


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


Parâmetros totais:         431,042
Parâmetros treináveis:     430,082
Parâmetros não treináveis: 960
Parâmetros adicionados:    0
Aumento vs. baseline:      0.0000%
FLOPs:                     1,428,345,100
GFLOPs:                    1.4283
Inferência:                21.319 ± 2.274 ms/imagem
Mediana:                   20.737 ms/imagem
Throughput:                46.91 FPS

CBAM
Parâmetros totais:         475,554
Parâmetros treináveis:     474,594
Parâmetros não treináveis: 960
Parâmetros adicionados:    44,512
Aumento vs. baseline:      10.3266%
FLOPs:                     1,453,624,540
GFLOPs:                    1.4536
Inferência:                78.433 ± 12.554 ms/imagem
Mediana:                   72.360 ms/imagem
Throughput:                12.75 FPS

CoordAttention
Parâmetros totais:         442,690
Parâmetros treináveis:     441,666
Parâmetros não treináveis: 1,024
Parâmetros adicionados:    11,648
Aumento vs. baseline:      2.7023%
FLOPs:                     1,442,235,980
GFLOPs:     